# CMFD Coarse Mesh

The CMFD coarse mesh decides which error modes the coarse problem can represent and how much the
coarse solve costs. This tutorial compares the `coarse_mesh` types of `CMFDAcceleration` on the
pin-homogenized C5G7 core:

* `"identity"`, one coarse cell per transport cell,
* `{"type": "reactor_lattice", ...}`, one coarse cell per lattice cell,
* `{"type": "cartesian", ...}`, one coarse cell per box of a Cartesian grid,
* `{"type": "logical_volumes", ...}`, one coarse cell per logical volume,
* `"local_aggregation"` (the default) or `{"type": "local_aggregation", "size": n}`, connected
  aggregates of about `n` transport cells built on each MPI rank,
* `{"type": "global_aggregation", "size": n}`, connected aggregates that may span MPI ranks.

A string selects a type with its defaults, and a dict selects a type and its parameters. The
coarse mesh is a logical grouping of transport cells. It does not change the transport mesh, its
partitioning or the transport solve. Coarse cells may contain several materials. CMFD homogenizes
their cross sections with the transport flux of the current iterate, which preserves the transport
reaction rates.

## The C5G7 model

The C5G7 benchmark [Lewis et al., NEA/NSC/DOC(2003)16] is a quarter core of four 17 x 17 pin
assemblies and a water reflector. The UO$_2$ assemblies sit on the diagonal. The MOX assemblies
have three enrichment zones (4.3%, 7.0% and 8.7%). Every assembly has 24 guide tubes and a central
fission chamber. The pin pitch is 1.26 cm and the reflector is one assembly (21.42 cm) thick. The
core center is in the lower-left corner, so the left and bottom boundaries are reflecting and the
right and top boundaries are vacuum. The benchmark provides seven-group cross sections with
upscattering in the four thermal groups.

The tutorials in this series use two models of that core.

* The **pin-homogenized model** mixes each fuel pin with its moderator by volume, so every pin
  cell is one 1.26 cm square mesh cell. It keeps the assembly layout, the MOX zoning, the guide
  tubes and the reflector, which are what make C5G7 hard for unaccelerated power iteration, and
  it solves in a few seconds. The option studies use it.
* The **heterogeneous model** in the [benchmark tutorial](c5g7_benchmark.ipynb) represents each
  pin by a square of the same area as the round pin, surrounded by moderator.

Both use a 2 x 2 KBA partition, so run the generated scripts with four MPI processes. A light
quadrature keeps the run times short. Production calculations need more directions.

## Model and measurement code

The next two cells build the C5G7 problem and measure a solve. They are the same in every tutorial
of this series. `solve()` builds a fresh problem for each case, so every solve starts from the
same state. Its `cmfd` argument holds the `CMFDAcceleration` options, and without it the power
iteration is unaccelerated. Its `solver` argument holds further `PowerIterationKEigenSolver`
options, such as `sweeps_per_update`. It returns the eigenvalue, the number of power iterations
(outers), the number of transport sweeps, the number of CMFD coarse cells, the number of skipped
CMFD corrections, whether the solve converged, the wall time, and the fission rate of every fuel
pin normalized to the average fuel pin. The wall time covers building the problem, initializing the
solver, and the solve. Building the problem is the same in every case.

In [ ]:
import math
from pathlib import Path

from mpi4py import MPI
from pyopensn.aquad import GLCProductQuadrature2DXY
from pyopensn.context import Finalize
from pyopensn.fieldfunc import FieldFunctionGridBased
from pyopensn.math import Vector3
from pyopensn.mesh import KBAGraphPartitioner, OrthogonalMeshGenerator
from pyopensn.solver import (
    CMFDAcceleration,
    DiscreteOrdinatesProblem,
    PowerIterationKEigenSolver,
)
from pyopensn.xs import MultiGroupXS

comm = MPI.COMM_WORLD
rank = comm.rank
tutorial_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
xs_dir = next(
    path / "test/assets/xs/c5g7"
    for path in (tutorial_dir, *tutorial_dir.parents)
    if (path / "test/assets/xs/c5g7").is_dir()
)

In [ ]:
PITCH = 1.26  # pin pitch (cm)
PIN_RADIUS = 0.54  # fuel pin radius (cm)
ASSEMBLY_PINS = 17
CORE_PINS = 2 * ASSEMBLY_PINS  # fuel-assembly pins per side of the quarter core
DOMAIN_PINS = 3 * ASSEMBLY_PINS  # including the water reflector

WATER, UO2, MOX_43, MOX_70, MOX_87, GUIDE_TUBE, FISSION_CHAMBER = range(7)
XS_FILES = [
    "XS_water",
    "XS_UO2",
    "XS_4_3pMOX",
    "XS_7pMOX",
    "XS_8_7pMOX",
    "XS_guide_tube",
    "XS_fission_chamber",
]
FUEL = (UO2, MOX_43, MOX_70, MOX_87)
GUIDE_TUBES = {
    (2, 5), (2, 8), (2, 11), (3, 3), (3, 13), (5, 2), (5, 5), (5, 8), (5, 11), (5, 14),
    (8, 2), (8, 5), (8, 11), (8, 14), (11, 2), (11, 5), (11, 8), (11, 11), (11, 14),
    (13, 3), (13, 13), (14, 5), (14, 8), (14, 11),
}
# One quarter of the MOX assembly, mirrored about its center row and column.
# 1, 2 and 3 are 4.3%, 7.0% and 8.7% MOX, G is a guide tube and F the fission chamber.
MOX_QUARTER = [
    "111111111",
    "122222222",
    "12222G22G",
    "122G23333",
    "122233333",
    "12G33G33G",
    "122333333",
    "122333333",
    "12G33G33F",
]
MOX_ZONES = {"1": MOX_43, "2": MOX_70, "3": MOX_87}


def pin_material(px, py):
    """Return the material of the pin cell in column px and row py of the domain."""
    if px >= CORE_PINS or py >= CORE_PINS:
        return WATER
    i, j = py % ASSEMBLY_PINS, px % ASSEMBLY_PINS
    if (i, j) == (8, 8):
        return FISSION_CHAMBER
    if (i, j) in GUIDE_TUBES:
        return GUIDE_TUBE
    if px // ASSEMBLY_PINS == py // ASSEMBLY_PINS:
        return UO2
    return MOX_ZONES[MOX_QUARTER[min(i, 16 - i)][min(j, 16 - j)]]


def load_cross_sections(homogenized):
    """Load the seven C5G7 materials, mixed with water over the pin cell if homogenized."""
    cross_sections = []
    for name in XS_FILES:
        xs = MultiGroupXS()
        xs.LoadFromOpenSn(str(xs_dir / f"{name}.xs"))
        cross_sections.append(xs)
    if not homogenized:
        return cross_sections
    fuel_fraction = math.pi * PIN_RADIUS**2 / PITCH**2
    water = cross_sections[WATER]
    return [water] + [
        MultiGroupXS.Combine([(xs, fuel_fraction), (water, 1.0 - fuel_fraction)])
        for xs in cross_sections[1:]
    ]


def make_mesh(homogenized):
    """Build the quarter core on a 2 x 2 KBA partition, so run with four MPI processes."""
    if homogenized:
        nodes = [PITCH * p for p in range(DOMAIN_PINS + 1)]
    else:
        side = math.sqrt(math.pi) * PIN_RADIUS  # square pin with the area of the round one
        gap = 0.5 * (PITCH - side)
        nodes = [PITCH * p + offset for p in range(CORE_PINS) for offset in (0.0, gap, gap + side)]
        nodes += [PITCH * (CORE_PINS + 0.5 * k) for k in range(2 * ASSEMBLY_PINS + 1)]
    cut = PITCH * ASSEMBLY_PINS
    mesh = OrthogonalMeshGenerator(
        node_sets=[nodes, nodes],
        partitioner=KBAGraphPartitioner(nx=2, ny=2, xcuts=[cut], ycuts=[cut]),
    ).Execute()
    mesh.SetOrthogonalBoundaries()

    def block_id(centroid, old_block_id):
        px, py = int(centroid.x // PITCH), int(centroid.y // PITCH)
        if not homogenized:
            local_x, local_y = centroid.x - px * PITCH, centroid.y - py * PITCH
            if not (gap < local_x < gap + side and gap < local_y < gap + side):
                return WATER
        return pin_material(px, py)

    mesh.SetBlockIDFromFunction(block_id)
    return mesh


def make_problem(
    homogenized=True,
    n_azimuthal=4,
    group_ranges=((0, 6),),
    inner_linear_method="classic_richardson",
    l_max_its=4,
    adjoint=False,
    verbose=False,
):
    """Build the C5G7 quarter-core transport problem."""
    cross_sections = load_cross_sections(homogenized)
    quadrature = GLCProductQuadrature2DXY(
        n_polar=2, n_azimuthal=n_azimuthal, scattering_order=0
    )
    problem = DiscreteOrdinatesProblem(
        mesh=make_mesh(homogenized),
        num_groups=7,
        groupsets=[
            {
                "groups_from_to": group_range,
                "angular_quadrature": quadrature,
                "inner_linear_method": inner_linear_method,
                "l_max_its": l_max_its,
                "l_abs_tol": 1.0e-10,
            }
            for group_range in group_ranges
        ],
        xs_map=[{"block_ids": [m], "xs": xs} for m, xs in enumerate(cross_sections)],
        boundary_conditions=[
            {"name": "xmin", "type": "reflecting"},
            {"name": "ymin", "type": "reflecting"},
        ],
        options={
            "adjoint": adjoint,
            "use_precursors": False,
            "power_default_kappa": 1.0,
            "verbose_inner_iterations": False,
            "verbose_outer_iterations": verbose,
        },
    )
    return problem, cross_sections

In [ ]:
def pin_fission_rates(problem, cross_sections):
    """Return the fission rate of every fuel pin, normalized to the average fuel pin.

    Each pin center is the center of a rectangular cell, where the bilinear PWLD flux equals
    its cell average.
    """
    pins = [
        (px, py)
        for py in range(CORE_PINS)
        for px in range(CORE_PINS)
        if pin_material(px, py) in FUEL
    ]
    points = [Vector3((px + 0.5) * PITCH, (py + 0.5) * PITCH, 0.0) for px, py in pins]
    flux = FieldFunctionGridBased.GetPointValues(problem.GetScalarFluxFieldFunction(), points)
    rates = []
    for n, (px, py) in enumerate(pins):
        sigma_f = cross_sections[pin_material(px, py)].sigma_f
        rates.append(sum(sigma_f[g] * flux[g][n][0] for g in range(len(sigma_f))))
    mean = sum(rates) / len(rates)
    return {pin: rate / mean for pin, rate in zip(pins, rates)}


def solve(label, cmfd=None, solver=None, k_tol=1.0e-7, max_iters=400, **problem_options):
    """Build a fresh problem, solve it with power iteration, and return the metrics.

    cmfd holds the CMFDAcceleration options. Without it the power iteration is unaccelerated.
    solver holds further PowerIterationKEigenSolver options, such as sweeps_per_update.
    """
    comm.Barrier()
    start = MPI.Wtime()
    problem, cross_sections = make_problem(**problem_options)
    acceleration = None
    solver_options = {"problem": problem, "k_tol": k_tol, "max_iters": max_iters}
    solver_options.update(solver or {})
    if cmfd is not None:
        acceleration = CMFDAcceleration(problem=problem, **cmfd)
        solver_options["acceleration"] = acceleration
    power_iteration = PowerIterationKEigenSolver(**solver_options)
    power_iteration.Initialize()
    power_iteration.Execute()
    seconds = comm.allreduce(MPI.Wtime() - start, op=MPI.MAX)
    return {
        "label": label,
        "k": power_iteration.GetEigenvalue(),
        "outers": power_iteration.GetNumPowerIterations(),
        "sweeps": power_iteration.GetNumSweeps(),
        "converged": power_iteration.IsConverged(),
        "coarse_cells": acceleration.GetNumCoarseCells() if acceleration else 0,
        "skipped": acceleration.GetNumSkippedCorrections() if acceleration else 0,
        "seconds": seconds,
        "pins": pin_fission_rates(problem, cross_sections),
    }


def report(results, k_reference, pin_reference=None):
    """Print one row per solve and one stable key per metric for the regression tests."""
    if rank != 0:
        return
    header = f"{'case':<34}{'k_eff':>12}{'k error':>10}{'outers':>8}{'sweeps':>8}"
    header += f"{'coarse':>8}{'skips':>6}{'conv':>5}"
    header += f"{'pin error':>10}" if pin_reference else ""
    print(header + f"{'time (s)':>10}")
    for r in results:
        k_error = r["k"] - k_reference
        row = f"{r['label']:<34}{r['k']:>12.8f}{k_error:>10.1e}{r['outers']:>8}{r['sweeps']:>8}"
        row += f"{r['coarse_cells']:>8}{r['skipped']:>6}{int(r['converged']):>5}"
        if pin_reference:
            row += f"{max_pin_error(r, pin_reference):>10.1e}"
        print(row + f"{r['seconds']:>10.2f}")
    for r in results:
        key = r["label"].replace(" ", "_")
        print(f"{key} k-effective error={r['k'] - k_reference:.6e}")
        print(f"{key} converged={int(r['converged'])}")


def max_pin_error(result, pin_reference):
    """Largest difference of the normalized pin fission rates from the reference."""
    return max(abs(result["pins"][pin] - value) for pin, value in pin_reference.items())

## Compare the coarse meshes

Every case uses two coarse groups and one sweep per update, as in the
[basics tutorial](cmfd_basics.ipynb), and changes only the coarse mesh. The Cartesian and
logical-volume cases both group the pin cells into boxes of 2 x 2 pins, with a last row and column
of one pin because the domain is 51 pins wide. The reference eigenvalue is 1.188819461.

In [ ]:
from pyopensn.logvol import RPPLogicalVolume

K_REFERENCE = 1.188819461
PIN_LATTICE = {
    "type": "reactor_lattice",
    "origin": [0.0, 0.0],
    "pitch": [PITCH, PITCH],
    "shape": [DOMAIN_PINS, DOMAIN_PINS],
}
# Box edges every two pins, ending at the domain boundary.
EDGES = [PITCH * min(2 * i, DOMAIN_PINS) for i in range((DOMAIN_PINS + 1) // 2 + 1)]
BOXES = [
    RPPLogicalVolume(xmin=x0, xmax=x1, ymin=y0, ymax=y1, infz=True)
    for x0, x1 in zip(EDGES, EDGES[1:])
    for y0, y1 in zip(EDGES, EDGES[1:])
]

cases = [
    ("identity", "identity"),
    ("pin lattice", PIN_LATTICE),
    ("cartesian 2x2 pins", {"type": "cartesian", "x_edges": EDGES, "y_edges": EDGES}),
    ("logical volumes 2x2 pins", {"type": "logical_volumes", "volumes": BOXES}),
    ("local 4", {"type": "local_aggregation", "size": 4}),
    ("global 4", {"type": "global_aggregation", "size": 4}),
    ("local 16", {"type": "local_aggregation", "size": 16}),
    ("local 32", "local_aggregation"),
    ("global 32", {"type": "global_aggregation", "size": 32}),
]
results = [
    solve(label, cmfd={"coarse_mesh": coarse_mesh, "coarse_groups": 2})
    for label, coarse_mesh in cases
]
report(results, K_REFERENCE)

## Interpret the results

**Coarse-cell size.** Smaller coarse cells converge in fewer power iterations: 39 for pin cells,
76 for 2 x 2 pin boxes, 93 for aggregates of about 4 pin cells, 213 for 16 and 301 for 32. CMFD
estimates a 1.26 cm pin cell at up to about 1.3 transport mean free paths. Error that varies within a
coarse cell is left to the transport sweeps, and as the coarse cells become optically thick the
convergence rate per update approaches the scattering ratio. From aggregates of 4 pin cells on,
CMFD warns that some coarse cells are thicker than 3 mean free paths. The
[user guide](../../../userguide/cmfd_acceleration.rst#optically-thick-coarse-cells) explains this limit.

**Cost of the coarse solve.** Larger coarse cells make the coarse problem smaller and cheaper. On
this small mesh the coarse solve is a large part of the run time, so the 2 x 2 pin boxes and the
aggregates of 4 pin cells are faster in wall time than the pin lattice, although the pin lattice
needs about half the power iterations. On production
meshes, where the sweeps dominate, the pin lattice is the better choice, and its coarse solve is
still small next to a sweep.

**Lattice, Cartesian and logical-volume coarse meshes.** These classify every transport cell by
its centroid, so the coarse mesh does not depend on the MPI partitioning, and a coarse cell may
span ranks. Here the pin lattice gives the same coarse cells as `"identity"`, and the logical
volumes give the same coarse cells as the Cartesian grid, so each pair converges identically. For
the lattice and Cartesian coarse meshes, CMFD logs at setup the number of coarse cells and the
number of fine faces between coarse cells that do not lie on a coarse-mesh plane:

```text
CMFD overlay: 2601 coarse cells. 0 of 5100 coarse-interface fine faces do not lie on an overlay plane (area fraction 0).
```

A nonzero count means that the transport mesh does not conform to the coarse mesh. The coarse
currents are still exact sums of fine-face currents, but the coarse cells have ragged boundaries.
The reactor lattice suits regular pin lattices. The Cartesian grid also takes nonuniform edges, for
example for assembly gaps. Logical volumes describe regions of any shape, but every transport-cell
centroid must lie in exactly one volume.

**Local and global aggregation.** Aggregation needs no geometry information, so it is the general
choice for unstructured meshes, but it does not follow pin boundaries. `local_aggregation` never
forms a coarse cell across an MPI rank boundary, so the partition cuts the coarse mesh.
`global_aggregation` builds aggregates that may span ranks, which reduces the number of small
aggregates at rank boundaries. Each coarse cell then has one owning rank, and the other ranks send
it their flux integrals and currents. With four ranks of 289 to 1156 cells each, the difference is
small here, and global aggregation needs a few more power iterations. It can help when ranks own
few cells.

All coarse meshes converge to the same eigenvalue, because CMFD changes only how fast power
iteration converges, not its fixed point.

In [ ]:
by_label = {r["label"]: r for r in results}
for result in results:
    assert result["converged"]
    assert abs(result["k"] - K_REFERENCE) < 1.0e-6
assert by_label["pin lattice"]["outers"] == by_label["identity"]["outers"]
assert (
    by_label["logical volumes 2x2 pins"]["outers"]
    == by_label["cartesian 2x2 pins"]["outers"]
)
assert by_label["pin lattice"]["outers"] < by_label["cartesian 2x2 pins"]["outers"]
assert by_label["local 4"]["outers"] < by_label["local 16"]["outers"]
assert by_label["local 16"]["outers"] < by_label["local 32"]["outers"]

## Finalize (for Jupyter Notebook only)

In script mode, PyOpenSn handles finalization automatically. In a Jupyter kernel, finalize OpenSn before MPI.

In [ ]:
if "opensn_console" not in globals():
    from IPython import get_ipython

    if get_ipython() is not None:
        Finalize()
        MPI.Finalize()